# Predict repeat orders with AdventureWorks in Colab

Which customers who ordered in the past year will order again in the next 90 days? Train and score this binary classification task with local Kumo Engine; regression and multiclass queries are optional alternatives.

Use a Linux x86-64 Colab GPU runtime with Python 3.12 and local SSD scratch. Follow [Linux setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment) and check the [available Colab runtime](https://research.google.com/colaboratory/runtime-version-faq.html). GPU memory and disk vary by allocation; allow space for source/prepared files, the materialized graph and model artifacts.

Supply the four original exports from [Microsoft AdventureWorks](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script), whose samples use the MIT license. This notebook prepares a separate local copy. Mount Drive first if your source files are there; copy retained artifacts out of temporary `/content` storage before the runtime ends.


## Install Kumo Engine

Install once, restart the Colab runtime, then resume at configuration.

In [ ]:
import platform
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Choose a Python 3.12 Colab runtime; found {sys.version.split()[0]}.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Select a Linux x86-64 Colab runtime.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "kumo-engine"])
print("Restart the Colab runtime once, then resume at Configuration.")


## Configure paths and scoring

Choose source and local working paths, and select training or reload. The scoring date is derived from the supplied order history.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

MODE = "train"  # "train" or "reload" from a saved local bundle.
MODEL_BUNDLE_PATH = None  # For reload, set to Path("<path/to/model_bundle>").
SOURCE_DATA_DIR = Path("<path/to/adventureworks_csvs>")
BASE = Path("/content/kumo-workspace")  # Local runtime disk; not mounted Drive.
RAW_DATA_DIR = BASE / "raw"
PREPARED_DATA_DIR = BASE / "prepared"
SCRATCH_DIR = BASE / "scratch"
ARTIFACT_ROOT = BASE / "artifacts"
PREDICTION_TIME = None  # Derived from the supplied order dates during preparation.

SAVE_PREDICTIONS = True

if MODE not in {"train", "reload"}:
    raise ValueError("MODE must be 'train' or 'reload'.")
runtime_root = Path("/content").resolve()
for name, path in {"BASE": BASE, "SCRATCH_DIR": SCRATCH_DIR}.items():
    resolved = path.expanduser().resolve()
    if not resolved.is_relative_to(runtime_root):
        raise ValueError(f"{name} must be under /content.")
    existing_parent = resolved
    while not existing_parent.exists():
        existing_parent = existing_parent.parent
    if existing_parent.stat().st_dev != runtime_root.stat().st_dev:
        raise ValueError(f"{name} must use local Colab disk, not mounted Drive.")
for name, path in {"SOURCE_DATA_DIR": SOURCE_DATA_DIR, "BASE": BASE}.items():
    if "<" in str(path):
        raise ValueError(f"Set {name} to a path available inside the Colab runtime.")


## Initialize the local runtime

Configure local scratch and check GPU access before materializing the graph.

In [ ]:
# Create local scratch and retained-artifact directories, test write access, configure Kumo, and verify GPU access.
import csv
import json
import shutil
import tempfile
from importlib.metadata import version
from uuid import uuid4

import pandas as pd
import torch
from IPython.display import display

import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumo.engine import Edge

scratch_root = SCRATCH_DIR.expanduser().resolve()
scratch_root.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=str(scratch_root))
import sys
print("Python:", sys.version.split()[0])
print("Kumo:", kumo.__version__, "| GA package:", version("kumo-engine"))
print("Scratch:", scratch_root)
print("Free local disk GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))
if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU-backed Colab runtime before materializing the graph.")
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    print("GPU:", device.name, "VRAM GiB:", round(device.total_memory / 2**30, 1))

def disk_checkpoint(label):
    free_gib = round(shutil.disk_usage(scratch_root).free / 2**30, 2)
    print(f"Disk after {label}: {free_gib} GiB free")
    if free_gib < 5:
        raise RuntimeError("Less than 5 GiB remains in local Colab storage; stop before the next step.")


## Verify access to the four AdventureWorks files

SHA-256 checks are optional for separately obtained exports. The notebook still checks that the source and local copy have identical bytes; later cells validate the expected table shape.


In [ ]:
import hashlib
import shutil

# Optional: set a SHA-256 value for any file whose exact bytes must be pinned.
# Leave as None when using a separately obtained AdventureWorks export.
EXPECTED_DATA_SHA256 = {
    "Customer.csv": None,
    "Product.csv": None,
    "SalesOrderHeader.csv": None,
    "SalesOrderDetail.csv": None,
}

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not SOURCE_DATA_DIR.is_dir():
    raise FileNotFoundError(f"Source directory is not visible: {SOURCE_DATA_DIR}")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, expected_hash in EXPECTED_DATA_SHA256.items():
    source = SOURCE_DATA_DIR / name
    if not source.is_file():
        raise FileNotFoundError(f"Missing AdventureWorks source file: {source}")
    source_hash = sha256(source)
    if expected_hash is not None and source_hash != expected_hash:
        raise RuntimeError(f"{name} does not match its configured SHA-256 checksum.")
    destination = RAW_DATA_DIR / name
    if destination.exists():
        if sha256(destination) != source_hash:
            raise RuntimeError(f"Local file differs from source at {destination}; use a clean workspace.")
    elif source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
        if sha256(destination) != source_hash:
            raise RuntimeError(f"Copied file differs from source at {destination}.")
    print("Verified AdventureWorks file:", destination)


## Prepare the four headerless TSV exports as local CSV tables

The four source files stay unchanged. The prepared directory contains only selected columns from each file; no customer ID is added to `SalesOrderDetail` and no new graph relationship is created. Each input row is checked for its expected field count.

In [ ]:
# (expected raw field count, selected column name -> zero-based raw field index)
TABLE_LAYOUTS = {
    'Customer': (7, {
        'CustomerID': 0, 'StoreID': 2, 'TerritoryID': 3,
    }),
    'Product': (25, {
        'ProductID': 0, 'Name': 1, 'Color': 5, 'ProductLine': 15,
        'Class': 16, 'StandardCost': 8, 'ListPrice': 9,
    }),
    'SalesOrderHeader': (26, {
        'SalesOrderID': 0, 'CustomerID': 10, 'OrderDate': 2,
        'OnlineOrderFlag': 6, 'TerritoryID': 12,
        'SubTotal': 19, 'TotalDue': 22,
    }),
    'SalesOrderDetail': (11, {
        'SalesOrderDetailID': 1, 'SalesOrderID': 0, 'ProductID': 4,
        'ModifiedDate': 10, 'OrderQty': 3, 'UnitPrice': 6,
        'LineTotal': 8,
    }),
}

PREPARED_DATA_DIR.mkdir(parents=True, exist_ok=True)
row_counts = {}
date_bounds = [None, None]
for table_name, (expected_fields, columns) in TABLE_LAYOUTS.items():
    raw_path = RAW_DATA_DIR / f'{table_name}.csv'
    prepared_path = PREPARED_DATA_DIR / f'{table_name}.csv'
    if not raw_path.is_file():
        raise FileNotFoundError(f'Missing uploaded table: {raw_path}')
    temporary_path = prepared_path.with_suffix('.csv.tmp')
    row_count = 0
    with raw_path.open(newline='', encoding='utf-8-sig') as raw_file, \
         temporary_path.open('w', newline='', encoding='utf-8') as prepared_file:
        reader = csv.reader(raw_file, delimiter='\t')
        writer = csv.writer(prepared_file)
        writer.writerow(columns)
        for row_count, row in enumerate(reader, start=1):
            if len(row) != expected_fields:
                raise ValueError(f'{raw_path.name} row {row_count}: expected '
                                 f'{expected_fields} fields, found {len(row)}')
            selected = [row[index] for index in columns.values()]
            if table_name == 'SalesOrderHeader':
                order_date = row[2][:10]
                date_bounds[0] = min(date_bounds[0], order_date) if date_bounds[0] else order_date
                date_bounds[1] = max(date_bounds[1], order_date) if date_bounds[1] else order_date
            for date_column in ('OrderDate', 'ModifiedDate'):
                if date_column in columns:
                    selected[list(columns).index(date_column)] = row[columns[date_column]][:10]
            writer.writerow(selected)
    if row_count == 0:
        raise ValueError(f'{raw_path.name} is empty')
    temporary_path.replace(prepared_path)
    row_counts[table_name] = row_count
print('Prepared rows:', row_counts)
print('OrderDate range:', *date_bounds)
PREDICTION_TIME = datetime.fromisoformat(date_bounds[1]) + timedelta(days=1)
print('Scoring boundary (day after latest OrderDate):', PREDICTION_TIME)
connector = kumo.FileConnector(root_dir=str(PREPARED_DATA_DIR))
print('Local tables:', connector.table_names())

## Connect to the local files

Inspect the prepared order columns through the local file connector.

In [ ]:
connector = kumo.FileConnector(root_dir=str(PREPARED_DATA_DIR))
print("Local tables:", connector.table_names())
display(pd.DataFrame([
    {"column": column.name, "physical_type": column.dtype, "semantic_type": column.stype}
    for column in connector["SalesOrderHeader"].columns
]))


## Define the AdventureWorks tables

Declare customer/product keys and feature roles, then connect order records using their original identifiers.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector['Customer'],
        ['CustomerID', 'StoreID', 'TerritoryID'],
        primary_key='CustomerID',
    )
    .with_column('StoreID', stype=Stype.categorical)
    .with_column('TerritoryID', stype=Stype.categorical)
)

product = (
    kumo.Table.from_source_table(
        connector['Product'],
        ['ProductID', 'Name', 'Color', 'ProductLine', 'Class',
         'StandardCost', 'ListPrice'],
        primary_key='ProductID',
    )
    .with_column('Name', stype=Stype.text)
    .with_column('Color', stype=Stype.categorical)
    .with_column('ProductLine', stype=Stype.categorical)
    .with_column('Class', stype=Stype.categorical)
    .with_column('StandardCost', stype=Stype.numerical)
    .with_column('ListPrice', stype=Stype.numerical)
)

sales_order_header = (
    kumo.Table.from_source_table(
        connector['SalesOrderHeader'],
        ['SalesOrderID', 'CustomerID', 'OrderDate', 'OnlineOrderFlag',
         'TerritoryID', 'SubTotal', 'TotalDue'],
        primary_key='SalesOrderID', time_column='OrderDate',
    )
    .with_column('CustomerID', stype=Stype.ID)
    .with_column('OrderDate', dtype=Dtype.date)
    .with_column('OnlineOrderFlag', stype=Stype.categorical)
    .with_column('TerritoryID', stype=Stype.categorical)
    .with_column('SubTotal', stype=Stype.numerical)
    .with_column('TotalDue', stype=Stype.numerical)
)

`SalesOrderDetail` uses its original `ModifiedDate` as the line-item time column. The prepared file retains only columns selected from the source; verify the timestamp's meaning before treating model metrics as a benchmark.

In [ ]:
sales_order_detail = (
    kumo.Table.from_source_table(
        connector['SalesOrderDetail'],
        ['SalesOrderDetailID', 'SalesOrderID', 'ProductID', 'ModifiedDate',
         'OrderQty', 'UnitPrice', 'LineTotal'],
        primary_key='SalesOrderDetailID', time_column='ModifiedDate',
    )
    .with_column('SalesOrderID', stype=Stype.ID)
    .with_column('ProductID', stype=Stype.ID)
    .with_column('ModifiedDate', dtype=Dtype.date)
    .with_column('OrderQty', stype=Stype.numerical)
    .with_column('UnitPrice', stype=Stype.numerical)
    .with_column('LineTotal', stype=Stype.numerical)
)

## Build the graph

Validate the customer–order–product relationships before creating the local snapshot.

In [ ]:
graph = kumo.Graph(
    tables=[customer, product, sales_order_header, sales_order_detail],
    edges=[
        Edge(src_table='SalesOrderHeader', fkey='CustomerID', dst_table='Customer'),
        Edge(src_table='SalesOrderDetail', fkey='SalesOrderID', dst_table='SalesOrderHeader'),
        Edge(src_table='SalesOrderDetail', fkey='ProductID', dst_table='Product'),
    ],
)
graph.validate()

## Inspect the graph

Inspect the actual graph’s tables and relationships before materialization. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because it is unavailable in the supported API.

In [ ]:
# graph.visualize(show_cols=True)

In [ ]:
print('Graph ID:', graph.id)
print('Tables:', list(graph._named_tables))
print('Edges:', len(graph.edges))
disk_checkpoint('graph validation')

### Materialize the graph

Read the prepared tables into a local snapshot for training and scoring.


In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)
disk_checkpoint("snapshot")

## Define the predictive query

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context.

The active query predicts whether a customer who ordered in the past year will order again in the next 90 days. It uses the existing `SalesOrderHeader.CustomerID → Customer.CustomerID` relationship and does not modify `SalesOrderDetail`.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        'PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) > 0 '
        'FOR EACH Customer.CustomerID '
        'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
    ),
)
pquery.validate()
print('Task type:', pquery.get_task_type())
print('Prediction time:', PREDICTION_TIME)

### Optional task alternatives

The next two query cells are commented-out alternatives. To switch tasks, comment out the active query above and uncomment exactly one alternative. Keep `MODE = "train"` when changing tasks because an existing model bundle belongs to its original query.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         'PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) '
#         'FOR EACH Customer.CustomerID '
#         'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
#     ),
# )
# pquery.validate()
# print('Task type:', pquery.get_task_type())

In [ ]:
# Multiclass classification: predict the territory of an active customer's next order.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         'PREDICT FIRST(SalesOrderHeader.TerritoryID, 0, 90, days) '
#         'FOR EACH Customer.CustomerID '
#         'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
#     ),
# )
# pquery.validate()
# print('Task type:', pquery.get_task_type())


### Generate a training table

Generate historical examples for the selected query; the default is 90-day repeat-order classification. Kumo’s temporal split keeps later label windows for evaluation. Inspect the split configuration in the model plan; the reload route skips generation.

In [ ]:
train_table = None
if MODE == "train":
    current_snapshot = globals().get("snapshot")
    snapshot_graph_id = getattr(current_snapshot, "id", "").split("@", 1)[0]
    if snapshot_graph_id != graph.id:
        print("Refreshing stale or missing graph snapshot...")
        snapshot = graph.snapshot()
    assert snapshot.id.split("@", 1)[0] == graph.id
    print("Graph:", graph.id)
    print("Snapshot:", snapshot.id)
    train_table = pquery.generate_training_table(snapshot)
    print("Training table:", train_table.train_table_path)
    disk_checkpoint("training-table generation")
else:
    print("Reload route: training-table generation skipped.")


## Configure the trainer

For `MODE = "train"`, the suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting. For `MODE = "reload"`, load the retained model bundle instead.

In [ ]:
from kumoapi.model_plan import RunMode

if MODE == "train":
    model_plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
    # Create a Trainer with your model plan.
    trainer = kumo.Trainer(model_plan)
    print(model_plan)
    print("Validation selection metric:", getattr(
        model_plan.training_job, "tune_metric", "not exposed by this build"))
else:
    if not MODEL_BUNDLE_PATH or not Path(MODEL_BUNDLE_PATH).is_dir():
        raise ValueError("Set MODEL_BUNDLE_PATH to a retained local model bundle directory.")
    trainer = kumo.Trainer.load(Path(MODEL_BUNDLE_PATH))
    print("Loaded:", MODEL_BUNDLE_PATH)


### Train the model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. Then retain the complete model bundle outside the temporary working directory.

In [ ]:
training_result = None
retained_bundle = Path(MODEL_BUNDLE_PATH) if MODE == "reload" else None
if MODE == "train":
    training_result = trainer.fit(train_table, snapshot=snapshot)
    disk_checkpoint("model fit")
    retained_bundle = ARTIFACT_ROOT / f"model-{uuid4().hex}"
    shutil.copytree(Path(training_result.bundle_dir), retained_bundle)
    print("Retained local bundle:", retained_bundle)
else:
    print("Reload route: fit skipped.")


## Inspect model metrics

Review the held-out metrics for the chosen task. AUROC measures binary ranking; regression MAE is an error in order counts. Reserve test metrics for final assessment.

In [ ]:
if training_result is None:
    print("Reloaded from:", retained_bundle)
    print("Saved training-table plan:", trainer.training_table_generation_plan)
else:
    print("Available metrics:", training_result.metrics)
    print("Reusable local bundle:", retained_bundle)
    for name, path in training_result.artifact_paths.items():
        print(name, path)
print("Download the retained bundle before Colab recycles the runtime.")


## Batch predictions

Prepare customers with an order in the preceding year at the boundary derived from the latest supplied `OrderDate`. These are future predictions, not an accuracy evaluation: the supplied extract has no outcomes beyond that boundary. Reuse the fitted trainer’s training-table plan and a timezone-naive UTC anchor.

In [ ]:
print('OrderDate range:', *date_bounds)
print('Configured prediction boundary:', PREDICTION_TIME)
print('Population: customers with an order in the preceding 365 days (90-day repeat-order classification)')

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)
print('Scoring population rows:', prediction_table.row_count)
if not prediction_table.row_count:
    raise RuntimeError(
        'No customers qualify for scoring at the derived boundary. '
        'Check the order dates, customer links, and past-365-day filter.'
    )
disk_checkpoint('prediction-table generation')

In [ ]:
from kumoapi.task import TaskType

prediction_options = {}
if pquery.get_task_type() == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5

elif pquery.get_task_type() == TaskType.MULTICLASS_CLASSIFICATION:
    prediction_options["num_classes_to_return"] = 1


### Score the customers

Run the selected model on the label-free population. The default binary decision threshold is 0.5.

In [ ]:
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    output_config=None, **prediction_options,
)


### Inspect and save results

Expect one row per customer. `True_PROB` is the repeat-order probability and `TARGET_PRED` its decision; the regression alternative predicts an order count, and multiclass returns its top territory as `CLASS` with its `SCORE`. Verify IDs before retaining the SDK’s Parquet files under `ARTIFACT_ROOT`.


In [ ]:
population = pd.read_parquet(prediction_table.table_path)
assert len(population) >= 2
assert not any(str(name).startswith("TARGET") for name in population)
batch_scores = prediction_result.frame
assert set(batch_scores.ENTITY) == set(population.ENTITY1)
assert len(batch_scores) == len(population)
assert batch_scores.ENTITY.is_unique
print(batch_scores.head(10).to_string(index=False))
disk_checkpoint("prediction")


In [ ]:
if SAVE_PREDICTIONS:
    prediction_dir = ARTIFACT_ROOT / f"predictions-{uuid4().hex}"
    prediction_dir.mkdir(parents=True)
    saved_files = []
    for index, source in enumerate(prediction_result.files):
        target = prediction_dir / f"part-{index}.parquet"
        shutil.copy2(source, target)
        saved_files.append(target)
    manifest = {
        "kumo_engine_version": version("kumo-engine"),
        "prediction_time": PREDICTION_TIME.isoformat(),
        "prediction_rows": prediction_result.num_rows,
        "model_bundle": str(retained_bundle),
                "prediction_files": [str(path) for path in saved_files],
    }
    (prediction_dir / "run.json").write_text(json.dumps(manifest, indent=2, default=str))
    print("Saved prediction directory:", prediction_dir)
    print("Download predictions and the model bundle before the Colab runtime ends.")
else:
    print("SDK prediction Parquet remains in local scratch for this runtime.")


## Reuse the model

Copy the complete retained model directory and prediction directory to Drive or download them before Colab stops. To score again, set `MODE="reload"` and `MODEL_BUNDLE_PATH`, then rebuild the compatible graph and snapshot. See [model reuse](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).
